# Decomposition Loss Weight Sweep — Local Windows

Same experiments as `colab_train_sample_eval2.ipynb` but runs locally.  
Requires the project to be at `c:\Users\ameli\Desktop\TezBaselines\MyCode`.

In [1]:
import os, sys
from pathlib import Path

REPO_PATH = r'c:\Users\ameli\Desktop\TezBaselines\MyCode'
assert Path(REPO_PATH).exists(), f'Not found: {REPO_PATH}'

os.chdir(REPO_PATH)
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'Memory  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Working directory: {os.getcwd()}')

PyTorch : 2.11.0+cu126
CUDA    : True
GPU     : NVIDIA GeForce RTX 4070
Memory  : 12.9 GB
Working directory: c:\Users\ameli\Desktop\TezBaselines\MyCode


In [2]:
import wandb, os
os.environ['WANDB_API_KEY'] = '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'

_api_key = os.environ.get('WANDB_API_KEY')
if _api_key:
    wandb.login(key=_api_key, relogin=False)
else:
    wandb.login()

print('wandb version:', wandb.__version__)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\ameli\_netrc
wandb: Currently logged in as: a-meliksahdemir (a-meliksahdemir-bo-azi-i-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb version: 0.27.0


In [ ]:
import torch, gc, importlib
from pathlib import Path

import train_with_mode as _twm
importlib.reload(_twm)
from train_with_mode import train

import evaluate_unified as _eu
importlib.reload(_eu)
from evaluate_unified import evaluate

DEVICE        = 'cuda' if torch.cuda.is_available() else 'cpu'
WANDB_PROJECT = 'diffusion-timeseries'
NUM_WORKERS   = 0     # must be 0 on Windows
SEED          = 42
BATCH_SIZE    = 64

NUM_EPOCHS          = 2000
LR                  = 1e-4
EVAL_METRICS_EVERY  = 200
N_METRIC_ITERATIONS = 3

# ── Experiment list ────────────────────────────────────────────────────────────
# 4 loss combinations × 2 model sizes = 8 experiments
# Large  : h256 / l8    Medium : h128 / l6

EXPERIMENTS = [

    # ── LARGE  (h256 / l8) ────────────────────────────────────────────────────
    dict(name='large_fft_trend_local',    group='large',
         hidden_dim=256, num_layers=8,
         fft_weight=1.0, trend_weight=1.0, season_weight=0.0),

    dict(name='large_trend_season_local', group='large',
         hidden_dim=256, num_layers=8,
         fft_weight=0.0, trend_weight=1.0, season_weight=1.0),

    dict(name='large_fft_season_local',   group='large',
         hidden_dim=256, num_layers=8,
         fft_weight=1.0, trend_weight=0.0, season_weight=1.0),

    dict(name='large_all_three_local',    group='large',
         hidden_dim=256, num_layers=8,
         fft_weight=1.0, trend_weight=1.0, season_weight=1.0),

    # ── MEDIUM  (h128 / l6) ───────────────────────────────────────────────────
    dict(name='medium_fft_trend_local',    group='medium',
         hidden_dim=128, num_layers=6,
         fft_weight=1.0, trend_weight=1.0, season_weight=0.0),

    dict(name='medium_trend_season_local', group='medium',
         hidden_dim=128, num_layers=6,
         fft_weight=0.0, trend_weight=1.0, season_weight=1.0),

    dict(name='medium_fft_season_local',   group='medium',
         hidden_dim=128, num_layers=6,
         fft_weight=1.0, trend_weight=0.0, season_weight=1.0),

    dict(name='medium_all_three_local',    group='medium',
         hidden_dim=128, num_layers=6,
         fft_weight=1.0, trend_weight=1.0, season_weight=1.0),
]

# ── Print plan ─────────────────────────────────────────────────────────────────
print(f'Device  : {DEVICE}')
print(f'Total   : {len(EXPERIMENTS)} experiments  |  epochs={NUM_EPOCHS}  eval_every={EVAL_METRICS_EVERY}  n_iter={N_METRIC_ITERATIONS}\n')
print(f"  {'#':<4} {'Name':<30} {'Group':<8} {'h':>5} {'l':>4} {'fft':>5} {'trend':>6} {'season':>7}")
print('  ' + '-'*68)
for i, exp in enumerate(EXPERIMENTS):
    print(f"  {i+1:<4} {exp['name']:<30} {exp['group']:<8}"
          f" {exp['hidden_dim']:>5} {exp['num_layers']:>4}"
          f" {exp['fft_weight']:>5.1f} {exp['trend_weight']:>6.1f} {exp['season_weight']:>7.1f}")
print()

# ── Run loop ───────────────────────────────────────────────────────────────────
for i, exp in enumerate(EXPERIMENTS):
    print(f"\n{'='*70}")
    print(f"  [{i+1}/{len(EXPERIMENTS)}]  {exp['name']}  (h={exp['hidden_dim']} l={exp['num_layers']})")
    print(f"{'='*70}\n")

    ckpt_dir  = f"output/ckpt_{exp['name']}"
    best_ckpt = f"{ckpt_dir}/best_model.pt"

    try:
        train(
            mode               = 'decomposition',
            device             = DEVICE,
            hidden_dim         = exp['hidden_dim'],
            num_layers         = exp['num_layers'],
            num_epochs         = NUM_EPOCHS,
            lr                 = LR,
            batch_size         = BATCH_SIZE,
            num_workers        = NUM_WORKERS,
            seed               = SEED,
            use_wandb          = True,
            wandb_project      = WANDB_PROJECT,
            wandb_run_name     = exp['name'],
            wandb_group        = exp['group'],
            eval_metrics       = True,
            eval_metrics_every = EVAL_METRICS_EVERY,
            n_metric_iterations= N_METRIC_ITERATIONS,
            img_pred_objective = 'pred_x0',
            img_loss_type      = 'l1',
            fft_weight         = exp['fft_weight'],
            trend_weight       = exp['trend_weight'],
            season_weight      = exp['season_weight'],
            checkpoint_dir     = ckpt_dir,
            finish_wandb       = False,
        )
    except Exception:
        import traceback
        print(f"\n!!! TRAINING FAILED: {exp['name']}")
        traceback.print_exc()
        try:
            import wandb
            if wandb.run is not None: wandb.finish()
        except Exception:
            pass
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        continue

    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    if Path(best_ckpt).exists():
        print(f"\n--- Post-training evaluation: {exp['name']} ---")
        try:
            evaluate(
                mode               = 'decomposition',
                checkpoint_path    = best_ckpt,
                device             = DEVICE,
                num_samples        = 256,
                n_metric_iterations= N_METRIC_ITERATIONS,
                compute_context_fid= True,
                use_wandb          = True,
                wandb_project      = WANDB_PROJECT,
                wandb_run_name     = exp['name'],
                wandb_group        = exp['group'],
                output_dir         = ckpt_dir,
                hidden_dim         = exp['hidden_dim'],
                num_layers         = exp['num_layers'],
            )
        except Exception:
            import traceback
            print(f"\n!!! EVALUATION FAILED: {exp['name']}")
            traceback.print_exc()
            try:
                import wandb
                if wandb.run is not None: wandb.finish()
            except Exception:
                pass
    else:
        print(f"   [skip eval] best_model.pt not found at {best_ckpt}")
        try:
            import wandb
            if wandb.run is not None: wandb.finish()
        except Exception:
            pass

    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

print('\n' + '='*70)
print('  ALL EXPERIMENTS COMPLETE')
print('='*70)

In [ ]:
import torch, gc
from pathlib import Path
from train_with_mode import train
from evaluate_unified import evaluate

# ── Large config — dominant loss experiments ───────────────────────────────────
# Each experiment: one loss at 1.0, the other two at 0.5 or 0.25
# 3 × (one dominant) × 2 scales = 6 experiments, all h256/l8

DOMINANT_EXPERIMENTS = [

    # ── One loss dominant at 1.0, others at 0.5 ───────────────────────────────
    dict(name='large_fft1_rest05_local',    group='large_dominant_05',
         hidden_dim=256, num_layers=8,
         fft_weight=1.0, trend_weight=0.5, season_weight=0.5),

    dict(name='large_trend1_rest05_local',  group='large_dominant_05',
         hidden_dim=256, num_layers=8,
         fft_weight=0.5, trend_weight=1.0, season_weight=0.5),

    dict(name='large_season1_rest05_local', group='large_dominant_05',
         hidden_dim=256, num_layers=8,
         fft_weight=0.5, trend_weight=0.5, season_weight=1.0),

    # ── One loss dominant at 1.0, others at 0.25 ──────────────────────────────
    dict(name='large_fft1_rest025_local',    group='large_dominant_025',
         hidden_dim=256, num_layers=8,
         fft_weight=1.0, trend_weight=0.25, season_weight=0.25),

    dict(name='large_trend1_rest025_local',  group='large_dominant_025',
         hidden_dim=256, num_layers=8,
         fft_weight=0.25, trend_weight=1.0, season_weight=0.25),

    dict(name='large_season1_rest025_local', group='large_dominant_025',
         hidden_dim=256, num_layers=8,
         fft_weight=0.25, trend_weight=0.25, season_weight=1.0),
]

# ── Print plan ─────────────────────────────────────────────────────────────────
print(f'Total : {len(DOMINANT_EXPERIMENTS)} experiments  |  epochs={NUM_EPOCHS}  eval_every={EVAL_METRICS_EVERY}  n_iter={N_METRIC_ITERATIONS}\n')
print(f"  {'#':<4} {'Name':<30} {'Group':<22} {'fft':>5} {'trend':>6} {'season':>7}")
print('  ' + '-'*72)
for i, exp in enumerate(DOMINANT_EXPERIMENTS):
    print(f"  {i+1:<4} {exp['name']:<30} {exp['group']:<22}"
          f" {exp['fft_weight']:>5.2f} {exp['trend_weight']:>6.2f} {exp['season_weight']:>7.2f}")
print()

# ── Run loop ───────────────────────────────────────────────────────────────────
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

for i, exp in enumerate(DOMINANT_EXPERIMENTS):
    print(f"\n{'='*70}")
    print(f"  [{i+1}/{len(DOMINANT_EXPERIMENTS)}]  {exp['name']}  (group: {exp['group']})")
    print(f"{'='*70}\n")

    ckpt_dir  = f"output/ckpt_{exp['name']}"
    best_ckpt = f"{ckpt_dir}/best_model.pt"

    try:
        train(
            mode               = 'decomposition',
            device             = DEVICE,
            hidden_dim         = exp['hidden_dim'],
            num_layers         = exp['num_layers'],
            num_epochs         = NUM_EPOCHS,
            lr                 = LR,
            batch_size         = BATCH_SIZE,
            num_workers        = NUM_WORKERS,
            seed               = SEED,
            use_wandb          = True,
            wandb_project      = WANDB_PROJECT,
            wandb_run_name     = exp['name'],
            wandb_group        = exp['group'],
            eval_metrics       = True,
            eval_metrics_every = EVAL_METRICS_EVERY,
            n_metric_iterations= N_METRIC_ITERATIONS,
            img_pred_objective = 'pred_x0',
            img_loss_type      = 'l1',
            fft_weight         = exp['fft_weight'],
            trend_weight       = exp['trend_weight'],
            season_weight      = exp['season_weight'],
            checkpoint_dir     = ckpt_dir,
            finish_wandb       = False,
        )
    except Exception:
        import traceback
        print(f"\n!!! TRAINING FAILED: {exp['name']}")
        traceback.print_exc()
        try:
            import wandb
            if wandb.run is not None: wandb.finish()
        except Exception:
            pass
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        continue

    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    if Path(best_ckpt).exists():
        print(f"\n--- Post-training evaluation: {exp['name']} ---")
        try:
            evaluate(
                mode               = 'decomposition',
                checkpoint_path    = best_ckpt,
                device             = DEVICE,
                num_samples        = 256,
                n_metric_iterations= N_METRIC_ITERATIONS,
                compute_context_fid= True,
                use_wandb          = True,
                wandb_project      = WANDB_PROJECT,
                wandb_run_name     = exp['name'],
                wandb_group        = exp['group'],
                output_dir         = ckpt_dir,
                hidden_dim         = exp['hidden_dim'],
                num_layers         = exp['num_layers'],
            )
        except Exception:
            import traceback
            print(f"\n!!! EVALUATION FAILED: {exp['name']}")
            traceback.print_exc()
            try:
                import wandb
                if wandb.run is not None: wandb.finish()
            except Exception:
                pass
    else:
        print(f"   [skip eval] best_model.pt not found at {best_ckpt}")
        try:
            import wandb
            if wandb.run is not None: wandb.finish()
        except Exception:
            pass

    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

print('\n' + '='*70)
print('  DOMINANT LOSS EXPERIMENTS COMPLETE')
print('='*70)